<a href="https://colab.research.google.com/github/fortunegbetox-tech/Code-ayant-le-score-maxi/blob/main/Code_route_ayant_le_score_le_plus_grand.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pytorch-lightning efficientnet_pytorch -q

import os, zipfile, torch, torch.nn as nn, pandas as pd, numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import pytorch_lightning as pl
from efficientnet_pytorch import EfficientNet
from google.colab import drive, files

drive.mount('/content/drive', force_remount=True)
ROOT = Path("/content/drive/MyDrive")
EXTRACT_DIR = Path("/content/img_fast")

if not EXTRACT_DIR.exists():
    with zipfile.ZipFile(ROOT/"Images.zip", 'r') as z:
        z.extractall(EXTRACT_DIR)

def get_absolute_index(d):
    return {f.stem: str(f.absolute()) for f in d.rglob("*.tif") if "__MACOSX" not in str(f)}

class RoadDataset(Dataset):
    def __init__(self, df, idx, tf, is_test=False):
        self.df, self.idx, self.tf, self.is_test = df, idx, tf, is_test
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img_id = str(row['Image_ID'])
        path = self.idx.get(img_id)
        img = self.tf(Image.open(path).convert("RGB")) if path else torch.zeros((3, 300, 300))
        if self.is_test: return img, img_id
        return img, torch.tensor(float(row['Target']), dtype=torch.float32)

class RoadModel(pl.LightningModule):
    def __init__(self, steps_per_epoch):
        super().__init__()
        self.net = EfficientNet.from_pretrained('efficientnet-b3', num_classes=1)
        self.loss_fn = nn.BCEWithLogitsLoss()
        self.steps = steps_per_epoch
    def forward(self, x): return self.net(x)
    def training_step(self, batch, batch_idx):
        x, y = batch
        return self.loss_fn(self(x).view(-1), y)
    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-2)
        sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, epochs=15, steps_per_epoch=self.steps)
        return [opt], [{"scheduler": sch, "interval": "step"}]

def run_pipeline():
    idx = get_absolute_index(EXTRACT_DIR)
    train_df = pd.read_csv(ROOT/"Train.csv").dropna(subset=['Image_ID'])
    test_df = pd.read_csv(ROOT/"Test.csv")

    tf_train = T.Compose([T.Resize((300, 300)), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    tf_test = T.Compose([T.Resize((300, 300)), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

    train_loader = DataLoader(RoadDataset(train_df, idx, tf_train), batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
    test_loader = DataLoader(RoadDataset(test_df, idx, tf_test, is_test=True), batch_size=32, num_workers=2)

    model = RoadModel(len(train_loader))
    trainer = pl.Trainer(accelerator="gpu", devices=1, max_epochs=15, precision="16-mixed", logger=False, enable_checkpointing=False)

    trainer.fit(model, train_loader)

    model.eval()
    model.cuda()
    preds, ids = [], []
    with torch.no_grad():
        for x, img_id in test_loader:
            y_hat = torch.sigmoid(model(x.to(model.device)))
            preds.extend(y_hat.cpu().numpy().flatten())
            ids.extend(img_id)

    output_name = "submission_finale_prouvee.csv"
    pd.DataFrame({"Image_ID": ids, "Target": preds}).to_csv(output_name, index=False)
    files.download(output_name)

if __name__ == "__main__":
    run_pipeline()

Mounted at /content/drive
Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b3-5fb5a3c3.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b3-5fb5a3c3.pth


100%|██████████| 47.1M/47.1M [00:00<00:00, 141MB/s]


Loaded pretrained weights for efficientnet-b3


MisconfigurationException: No supported gpu backend found!